# 01 – Data Quality Assessment

## Objective

This notebook performs a structured data quality assessment of the raw credit application dataset prior to downstream analytical use.

The evaluation focuses on the following quality dimensions:

- **Completeness** – identification and quantification of missing values  
- **Consistency** – detection of inconsistent formats, categorical encodings, and schema variations  
- **Validity** – identification of impossible or out-of-range values  
- **Uniqueness** – detection of duplicate records and conflicting identifiers  

For each identified issue, we quantify its extent, assess its potential analytical impact, and apply justified remediation strategies where appropriate.

The outcome of this process is a cleaned and standardized dataset suitable for reliable fairness analysis and governance evaluation.

In [21]:
# Standard libraries
import json
import copy
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Display settings (ensuring all columns are visible)
pd.set_option("display.max_columns", None)

# Define data path
DATA_PATH = Path("../data/raw_credit_applications.json")

In [22]:
# Load raw JSON file
with open(DATA_PATH, "r") as f:
    records = json.load(f)

print(f"Total applications loaded: {len(records)}")

Total applications loaded: 502


In [23]:
# Inspect structure of first record
print("Top-level fields in first application record:")
print(records[0].keys())

Top-level fields in first application record:
dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])


In [24]:
# Validate structural consistency across all records

# Collect the set of top-level keys for each record
key_sets = [set(r.keys()) for r in records]

# Identify unique structural patterns
unique_structures = {tuple(sorted(keys)) for keys in key_sets}

print(f"Number of distinct top-level structures: {len(unique_structures)}")

Number of distinct top-level structures: 4


In [25]:
# Display distinct top-level structures

for i, structure in enumerate(unique_structures, 1):
    print(f"\nStructure {i}:")
    print(structure)


Structure 1:
('_id', 'applicant_info', 'decision', 'financials', 'spending_behavior')

Structure 2:
('_id', 'applicant_info', 'decision', 'financials', 'processing_timestamp', 'spending_behavior')

Structure 3:
('_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior')

Structure 4:
('_id', 'applicant_info', 'decision', 'financials', 'loan_purpose', 'spending_behavior')


### Top-Level Schema Variation

The dataset exhibits four distinct top-level schema variations.

A core schema structure is consistently present across all records, comprising:
- `_id`
- `applicant_info`
- `financials`
- `decision`
- `spending_behavior`

However, the following additional fields appear inconsistently:
- `loan_purpose`
- `processing_timestamp`
- `notes`

This indicates that certain attributes are optional and inconsistently populated across the dataset. Prior to transformation and analysis, the schema must be unified to ensure consistent representation of all expected fields.

In [26]:
# Quantify presence of optional top-level fields

optional_fields = ["loan_purpose", "processing_timestamp", "notes"]

total_records = len(records)

summary = []

for field in optional_fields:
    present = sum(field in r for r in records)
    summary.append({
        "field": field,
        "present_count": present,
        "present_pct (%)": round((present / total_records) * 100, 2)
    })

pd.DataFrame(summary).sort_values("present_count", ascending=False).reset_index(drop=True)

,field,present_count,present_pct (%)
0,processing_timestamp,62,12.35
1,loan_purpose,50,9.96
2,notes,2,0.40


### Schema Normalization

The dataset contains a consistent core structure, but several top-level fields appear only in subsets of records (e.g., `loan_purpose`, `processing_timestamp`, `notes`). 

To ensure consistent processing and reproducible transformation steps, we standardize the top-level schema by explicitly adding these optional fields to all records when absent and representing missing values as null. This does not imply that these fields are mandatory. It enforces a uniform schema representation prior to flattening and downstream analysis.

In [27]:
records_norm = copy.deepcopy(records)  # work on a copy; keep `records` as raw reference

In [28]:
# Define a canonical top-level schema for consistent processing

canonical_fields = [
    "_id",
    "applicant_info",
    "financials",
    "decision",
    "spending_behavior",
    "loan_purpose",
    "processing_timestamp",
    "notes",
]

In [29]:
# Normalize records_norm: ensure all canonical fields exist in every record
# Missing optional fields are added as explicit nulls (None)

for r in records_norm:
    for field in canonical_fields:
        if field not in r:
            r[field] = None

In [30]:
# Verify that all records now share the same top-level structure after normalization
normalized_structures = {tuple(sorted(r.keys())) for r in records_norm}

assert len(normalized_structures) == 1, "Schema normalization failed"
print(f"✓ All {len(records_norm)} records now share a unified top-level schema.")

✓ All 502 records now share a unified top-level schema.
